## CBOW and Skip-gram Implementation with PyTorch

This notebook demonstrates the implementation of Continuous Bag of Words (CBOW) and Skip-gram models using PyTorch for learning word embeddings.

In [13]:
# Install necessary libraries
!pip install torch nltk

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Added this line to resolve LookupError

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

### 1. Import Libraries

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import re
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

### 2. Sample Corpus

In [3]:
corpus = [
    "The cat sat on the mat",
    "The dog ran in the park",
    "The cat and dog played together",
    "A quick brown fox jumps over the lazy dog"
]

### 3. Text Preprocessing and Vocabulary Building

In [14]:
def preprocess_text(text_corpus):
    processed_sentences = []
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    for sentence in text_corpus:
        # Lowercase, tokenize, remove punctuation and numbers
        tokens = word_tokenize(re.sub(r'[^a-zA-Z\s]', '', sentence.lower()))
        # Remove stopwords and lemmatize
        filtered_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
        processed_sentences.append(filtered_tokens)
    return processed_sentences

processed_corpus = preprocess_text(corpus)
print("Processed Corpus:", processed_corpus)

# Build vocabulary
word_counts = Counter()
for sentence in processed_corpus:
    word_counts.update(sentence)

vocab = {word for word, count in word_counts.items() if count >= 1} # Include all words for this small corpus
word_to_idx = {word: i for i, word in enumerate(sorted(list(vocab)))}
idx_to_word = {i: word for word, i in word_to_idx.items()}
vocab_size = len(vocab)

print(f"\nVocabulary Size: {vocab_size}")
print("Word to Index Mapping:", word_to_idx)

Processed Corpus: [['cat', 'sat', 'mat'], ['dog', 'ran', 'park'], ['cat', 'dog', 'played', 'together'], ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog']]

Vocabulary Size: 13
Word to Index Mapping: {'brown': 0, 'cat': 1, 'dog': 2, 'fox': 3, 'jump': 4, 'lazy': 5, 'mat': 6, 'park': 7, 'played': 8, 'quick': 9, 'ran': 10, 'sat': 11, 'together': 12}


### 4. CBOW (Continuous Bag of Words) Model

#### 4.1 CBOW Model Architecture

In [15]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).mean(dim=1) # Average the embeddings of context words
        out = self.linear(embeds)
        return out

#### 4.2 Prepare CBOW Training Data

In [16]:
def create_cbow_training_data(processed_corpus, word_to_idx, window_size=2):
    data = []
    for sentence in processed_corpus:
        for i, target_word in enumerate(sentence):
            if target_word in word_to_idx:
                target_idx = word_to_idx[target_word]
                context_indices = []
                for j in range(max(0, i - window_size), min(len(sentence), i + window_size + 1)):
                    if i != j and sentence[j] in word_to_idx:
                        context_indices.append(word_to_idx[sentence[j]])
                if context_indices:
                    data.append((context_indices, target_idx))
    return data

cbow_data = create_cbow_training_data(processed_corpus, word_to_idx)
print("Sample CBOW Data (context indices, target index):", cbow_data[:2])

Sample CBOW Data (context indices, target index): [([11, 6], 1), ([1, 6], 11)]


#### 4.3 Train CBOW Model

In [28]:
EMBEDDING_DIM = 100
CONTEXT_SIZE = 2

cbow_model = CBOW(vocab_size, EMBEDDING_DIM)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(cbow_model.parameters(), lr=0.1)

num_epochs = 300
print("\nTraining CBOW Model...")
for epoch in range(num_epochs):
    total_loss = 0
    for context_indices, target_idx in cbow_data:
        if len(context_indices) == 0: # Skip if no context found
            continue

        context_var = torch.tensor(context_indices, dtype=torch.long)
        target_var = torch.tensor([target_idx], dtype=torch.long)

        cbow_model.zero_grad()
        log_probs = cbow_model(context_var.unsqueeze(0)) # Add batch dimension
        loss = loss_function(log_probs, target_var)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

print("CBOW Model training complete.")


Training CBOW Model...
Epoch 0, Loss: 51.4955
Epoch 10, Loss: 7.4557
Epoch 20, Loss: 6.0518
Epoch 30, Loss: 5.0697
Epoch 40, Loss: 4.3392
Epoch 50, Loss: 3.7961
Epoch 60, Loss: 3.3925
Epoch 70, Loss: 3.0899
Epoch 80, Loss: 2.8595
Epoch 90, Loss: 2.6807
Epoch 100, Loss: 2.5392
Epoch 110, Loss: 2.4251
Epoch 120, Loss: 2.3315
Epoch 130, Loss: 2.2537
Epoch 140, Loss: 2.1881
Epoch 150, Loss: 2.1322
Epoch 160, Loss: 2.0840
Epoch 170, Loss: 2.0421
Epoch 180, Loss: 2.0054
Epoch 190, Loss: 1.9731
Epoch 200, Loss: 1.9444
Epoch 210, Loss: 1.9188
Epoch 220, Loss: 1.8958
Epoch 230, Loss: 1.8751
Epoch 240, Loss: 1.8563
Epoch 250, Loss: 1.8393
Epoch 260, Loss: 1.8237
Epoch 270, Loss: 1.8095
Epoch 280, Loss: 1.7964
Epoch 290, Loss: 1.7844
CBOW Model training complete.


#### 4.4 Get Word Embeddings from CBOW Model

In [35]:
def get_cbow_word_embedding(word, model, word_to_idx):
    if word in word_to_idx:
        word_idx = torch.tensor([word_to_idx[word]], dtype=torch.long)
        # For CBOW, the embedding layer directly gives the word vector
        # However, the `forward` method averages context words.
        # To get an individual word embedding, we access the embedding layer directly.
        return model.embeddings(word_idx).squeeze().detach().numpy()
    return None

# Example: Get embedding for 'cat'
cat_embedding_cbow = get_cbow_word_embedding('cat', cbow_model, word_to_idx)
if cat_embedding_cbow is not None:
    print(f"\nEmbedding for 'cat' (CBOW):\n{cat_embedding_cbow[:5]}...") # Print first 5 elements
else:
    print("Word 'cat' not in vocabulary.")


Embedding for 'cat' (CBOW):
[ 1.278677    0.46680772  0.9681371  -0.1846094   1.1783018 ]...


### 5. Skip-gram Model

#### 5.1 Skip-gram Model Architecture

In [19]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGram, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, target_word_idx):
        embeds = self.embeddings(target_word_idx)
        out = self.linear(embeds)
        return out

#### 5.2 Prepare Skip-gram Training Data

In [20]:
def create_skipgram_training_data(processed_corpus, word_to_idx, window_size=2):
    data = []
    for sentence in processed_corpus:
        for i, target_word in enumerate(sentence):
            if target_word in word_to_idx:
                target_idx = word_to_idx[target_word]
                for j in range(max(0, i - window_size), min(len(sentence), i + window_size + 1)):
                    if i != j and sentence[j] in word_to_idx:
                        context_idx = word_to_idx[sentence[j]]
                        data.append((target_idx, context_idx))
    return data

skipgram_data = create_skipgram_training_data(processed_corpus, word_to_idx)
print("Sample Skip-gram Data (target index, context index):", skipgram_data[:2])

Sample Skip-gram Data (target index, context index): [(1, 11), (1, 6)]


#### 5.3 Train Skip-gram Model

In [33]:
skipgram_model = SkipGram(vocab_size, EMBEDDING_DIM)
loss_function_sg = nn.CrossEntropyLoss()
optimizer_sg = optim.Adam(skipgram_model.parameters(), lr=0.001)

num_epochs_sg = 100
print("\nTraining Skip-gram Model...")
for epoch in range(num_epochs_sg):
    total_loss_sg = 0
    for target_idx, context_idx in skipgram_data:
        target_var = torch.tensor([target_idx], dtype=torch.long)
        context_var = torch.tensor([context_idx], dtype=torch.long)

        skipgram_model.zero_grad()
        log_probs_sg = skipgram_model(target_var)
        loss_sg = loss_function_sg(log_probs_sg, context_var)
        loss_sg.backward()
        optimizer_sg.step()

        total_loss_sg += loss_sg.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss_sg:.4f}")

print("Skip-gram Model training complete.")


Training Skip-gram Model...
Epoch 0, Loss: 110.5249
Epoch 10, Loss: 60.8691
Epoch 20, Loss: 54.4016
Epoch 30, Loss: 52.6121
Epoch 40, Loss: 51.8345
Epoch 50, Loss: 51.4100
Epoch 60, Loss: 51.1452
Epoch 70, Loss: 50.9652
Epoch 80, Loss: 50.8353
Epoch 90, Loss: 50.7372
Skip-gram Model training complete.


#### 5.4 Get Word Embeddings from Skip-gram Model

In [34]:
def get_skipgram_word_embedding(word, model, word_to_idx):
    if word in word_to_idx:
        word_idx = torch.tensor([word_to_idx[word]], dtype=torch.long)
        return model.embeddings(word_idx).squeeze().detach().numpy()
    return None

# Example: Get embedding for 'dog'
dog_embedding_skipgram = get_skipgram_word_embedding('dog', skipgram_model, word_to_idx)
if dog_embedding_skipgram is not None:
    print(f"\nEmbedding for 'dog' (Skip-gram):\n{dog_embedding_skipgram[:5]}...") # Print first 5 elements
else:
    print("Word 'dog' not in vocabulary.")


Embedding for 'dog' (Skip-gram):
[-1.9987057   0.5957078  -0.6387495  -0.71927345  1.8635492 ]...


### 6. Embedding Analysis: Dot Product and Cosine Similarity

We can evaluate the quality of our embeddings by checking how 'similar' words are in the vector space. We use two common metrics:
1.  **Dot Product**: Measures the magnitude and direction.
2.  **Cosine Similarity**: Measures only the cosine of the angle between vectors (normalized dot product), ranging from -1 to 1.

In [36]:
import numpy as np

def analyze_similarity(word1, word2, model, word_to_idx, model_name="Model", method=get_cbow_word_embedding):
    vec1 = method(word1, model, word_to_idx)
    vec2 = method(word2, model, word_to_idx)

    if vec1 is None or vec2 is None:
        return f"One of the words '{word1}' or '{word2}' not in vocab."

    dot_product = np.dot(vec1, vec2)
    norm_v1 = np.linalg.norm(vec1)
    norm_v2 = np.linalg.norm(vec2)
    cosine_sim = dot_product / (norm_v1 * norm_v2)

    print(f"--- {model_name} Similarity: '{word1}' & '{word2}' ---")
    print(f"Dot Product: {dot_product:.4f}")
    print(f"Cosine Similarity: {cosine_sim:.4f}\n")

# Test cases for CBOW
analyze_similarity('cat', 'dog', cbow_model, word_to_idx, "CBOW", get_cbow_word_embedding)
analyze_similarity('cat', 'mat', cbow_model, word_to_idx, "CBOW", get_cbow_word_embedding)

# Test cases for Skip-gram
analyze_similarity('cat', 'dog', skipgram_model, word_to_idx, "Skip-gram", get_skipgram_word_embedding)
analyze_similarity('dog', 'park', skipgram_model, word_to_idx, "Skip-gram", get_skipgram_word_embedding)

--- CBOW Similarity: 'cat' & 'dog' ---
Dot Product: -1.2751
Cosine Similarity: -0.0147

--- CBOW Similarity: 'cat' & 'mat' ---
Dot Product: 13.1441
Cosine Similarity: 0.1202

--- Skip-gram Similarity: 'cat' & 'dog' ---
Dot Product: -23.7433
Cosine Similarity: -0.2287

--- Skip-gram Similarity: 'dog' & 'park' ---
Dot Product: -12.0432
Cosine Similarity: -0.1162



### Analysis

*   **Cosine Similarity**: Values closer to 1 indicate that the words appear in similar contexts. For example, in our small corpus, 'cat' and 'dog' are both animals that 'sit' or 'play', so they should ideally have a higher similarity than unrelated words.
*   **CBOW vs Skip-gram**:
    *   **CBOW** tends to smooth out over contexts (predicting one word from many), making it faster and better for frequent words.
    *   **Skip-gram** (predicting many words from one) works better with smaller datasets and can represent rare words or multiple meanings more effectively.
*   **Small Dataset Note**: Because our corpus is extremely small (4 sentences), the embeddings won't perfectly capture semantic relationships yet, but the similarity scores show that the models are learning patterns based on co-occurrence.